# RISK ANALYTICS — 01 Inspección de datos

**Fuente:** `/Volumes/eventosriesgoyvar/default/bdeventos/BD eventos riesgo operacional.csv`

**Alcance de esta etapa:** solo lectura. No se construye pipeline ni dashboard, no se modifican archivos existentes.

Inspección preliminar: separador `;`, encoding ISO-8859-1, fechas `dd/MM/yyyy` (algunas sin cero inicial, p. ej. `2/03/2026`), montos como texto con `.` de miles, ausentes representados como `-`, encabezados con espacios/duplicados (dos columnas `FUENTE`).

In [0]:
# Carga del CSV tal cual está: todo como string, sin inferencia, sin escribir nada
from pyspark.sql import functions as F

RUTA_CSV = "/Volumes/eventosriesgoyvar/default/bdeventos/BD eventos riesgo operacional.csv"

df_raw = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", False)
    .csv(RUTA_CSV))

print("Filas:", df_raw.count(), "| Columnas:", len(df_raw.columns))
display(df_raw.limit(5))

Filas: 51889 | Columnas: 58


FECHAINICIO,HORAINICIO,MINUTOINICIO,FECHAFINAL,HORAFIN,MINUTOFIN,FECHADESCUBRIMIENTO,HORADES,MINUTODES,TIPO,FECHA CONTABILIZACION,HORACON,MINUTOCON,FECHA_RECUPERACIÓN,HORAREC,MINUTOREC,MONEDA,CUANTIA_BRUTA,CUANTIA_TOTAL_RECUPERADA,CUANTIA_RECUPERADA_SEGUROS,CUANTIA_OTRAS_RECUPERACIONES,CUANTIA_NETA,COD CATEGORIA N1,CATEGORIA N1,CODIGO SUBCATEGORIA N2,SUBCATEGORIA N2,CODIGO SUBCATEGORIA N3,SUBCATEGORIA RIESGO N3,CODIGOPRODUCTO,PRODUCTO,CODIGOPERDIDA,CODIGOSISGEO,CIUDAD:,CODIGOSUCURSAL,SUCURSAL,. CANAL:,CODIGOIMPACTO,IMPACTO,CODIGOFUENTE,FUENTE39,CODIGO LINEA OPERATIVA N1,LINEA OPERATIVA N1,CODIGO LINEA OPERATIVA N2,LINEA OPERATIVA N2,CODESTROR,CODPROCESO,PROCESO,CODACTIVIADAD,ACTIVIDAD,CODRIESGO,CODCAUSA,CUENTA PÉRDIDA (8 digitos),DOCUMENTO_PÉRDIDA,CUENTA RECUPERACIÓN (8 digitos),DOCUMENTO_RECUPERACIÓN,CLASE DE RIESGO,Fuente56,IDINSUMO
05/01/2026,14,45,08/01/2026,15,30,05/01/2026,14,45,P,null,null,null,null,null,null,PESOS,400.000,-,-,-,400.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,11,CONFIDIARIO,B,15001000,TUNJA,72,TIBASOSA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6022236
13/01/2026,21,29,17/01/2026,11,44,13/01/2026,21,29,P,null,null,null,null,null,null,PESOS,600.000,-,-,-,600.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,05001000,MEDELLIN,20,CASTILLA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6045897
02/01/2026,09,25,25/02/2026,08,19,02/01/2026,09,25,P,null,null,null,null,null,null,PESOS,156.800,-,-,-,156.800,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,11001000,BOGOTA,98,LA SOLEDAD,POS,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,3",SERVICIOS DE TARJETAS DE CRÉDITO Y/O DÉBITO,1.1.9.4.3,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.14,POS,8,627,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6013564
05/01/2026,10,25,06/01/2026,17,49,05/01/2026,10,25,P,null,null,null,null,null,null,PESOS,140.000,-,-,-,140.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,05001000,MEDELLIN,39,BARRIO ANTIOQUIA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6019795
02/01/2026,10,53,06/01/2026,17,45,02/01/2026,10,53,P,null,null,null,null,null,null,PESOS,400.000,-,-,-,400.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,11,CONFIDIARIO,B,05001000,MEDELLIN,18,SAN CRISTOBAL,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6014557


In [0]:
import re

# Limpieza SOLO en memoria: nombres snake_case, trim, '-' y '' -> NULL
mapa_nombres = {c: re.sub(r"[^0-9a-zA-Z]+", "_", c.strip().lower()).strip("_") for c in df_raw.columns}
d = df_raw.select([F.trim(F.col("`" + c + "`")).alias(mapa_nombres[c]) for c in df_raw.columns])
for c in d.columns:
    d = d.withColumn(c, F.when(F.trim(F.col(c)).isin("-", ""), None).otherwise(F.trim(F.col(c))))
d.cache()
N = d.count()
print("Filas:", N, "| Columnas:", len(d.columns))

# Perfil de calidad: nulos y valores distintos por columna
prof = d.agg(*[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias("nulls_" + c) for c in d.columns]).collect()[0].asDict()
dist = d.agg(*[F.countDistinct(F.col(c)).alias("nd_" + c) for c in d.columns]).collect()[0].asDict()
perfil = spark.createDataFrame(
    [(c, int(prof["nulls_" + c]), round(100 * prof["nulls_" + c] / N, 1), int(dist["nd_" + c])) for c in d.columns],
    ["columna", "nulos", "pct_nulos", "valores_distintos"])
display(perfil)

Filas: 51889 | Columnas: 58


columna,nulos,pct_nulos,valores_distintos
fechainicio,0,0.0,4098
horainicio,0,0.0,24
minutoinicio,0,0.0,68
fechafinal,0,0.0,3798
horafin,0,0.0,25
minutofin,0,0.0,60
fechadescubrimiento,0,0.0,4093
horades,0,0.0,24
minutodes,0,0.0,68
tipo,0,0.0,3


In [0]:
# Granularidad: 1 fila = 1 evento de riesgo. Verificación de claves y duplicados
exact_dup = N - d.dropDuplicates().count()
nid = d.filter(F.col("idinsumo").isNotNull()).count()
nid_dist = d.select("idinsumo").distinct().count()
clave_compuesta = ["fechainicio", "horainicio", "minutoinicio", "ciudad", "codigoproducto", "codproceso", "cuantia_bruta"]
key_dist = d.select(*clave_compuesta).distinct().count()

print(f"Filas exactamente duplicadas: {exact_dup}")
print(f"IDINSUMO no nulos: {nid} | distintos: {nid_dist} | repetidos: {nid - nid_dist}")
print(f"Combinaciones distintas de clave compuesta (fecha+hora+min+ciudad+producto+proceso+bruta): {key_dist} (vs {N} filas)")

Filas exactamente duplicadas: 1031
IDINSUMO no nulos: 28863 | distintos: 28351 | repetidos: 512
Combinaciones distintas de clave compuesta (fecha+hora+min+ciudad+producto+proceso+bruta): 49363 (vs 51889 filas)


In [0]:
# Fechas en dd/MM/yyyy, algunas sin cero inicial -> parseo tolerante con try_to_date
def parse_fecha(colname):
    c = F.col(colname)
    return F.coalesce(
        F.try_to_date(c, "dd/MM/yyyy"), F.try_to_date(c, "d/M/yyyy"),
        F.try_to_date(c, "dd/M/yyyy"), F.try_to_date(c, "d/MM/yyyy"))

df = (d.withColumn("fecha_inicio", parse_fecha("fechainicio"))
       .withColumn("fecha_descubrimiento", parse_fecha("fechadescubrimiento"))
       .withColumn("fecha_final", parse_fecha("fechafinal"))
       .withColumn("anio", F.year("fecha_inicio")))

df.agg(F.min("fecha_inicio").alias("inicio"), F.max("fecha_inicio").alias("fin"),
       F.min("fecha_descubrimiento").alias("desc_min"), F.max("fecha_descubrimiento").alias("desc_max"),
       F.sum(F.when(F.col("fecha_inicio").isNull(), 1).otherwise(0)).alias("sin_parsear")).show(truncate=False)
display(df.groupBy("anio").count().orderBy("anio"))

+----------+----------+----------+----------+-----------+
|inicio    |fin       |desc_min  |desc_max  |sin_parsear|
+----------+----------+----------+----------+-----------+
|2014-12-05|2026-09-30|2014-12-22|2026-10-22|0          |
+----------+----------+----------+----------+-----------+



anio,count
2014,2
2015,6668
2016,6184
2017,5136
2018,4957
2019,4398
2020,4566
2021,3794
2022,3096
2023,3501


In [0]:
# Montos: texto con '.' de miles -> double (los '-' ya quedaron como NULL)
df = (df.withColumn("cuantia_bruta_num", F.regexp_replace(F.col("cuantia_bruta"), "\\.", "").cast("double"))
        .withColumn("cuantia_neta_num", F.regexp_replace(F.col("cuantia_neta"), "\\.", "").cast("double"))
        .withColumn("cuantia_recuperada_num", F.regexp_replace(F.col("cuantia_total_recuperada"), "\\.", "").cast("double")))

df.select(F.count("cuantia_bruta_num").alias("eventos_con_bruta"),
          F.min("cuantia_bruta_num").alias("bruta_min"), F.max("cuantia_bruta_num").alias("bruta_max"),
          F.sum("cuantia_bruta_num").alias("bruta_suma"),
          F.count("cuantia_neta_num").alias("eventos_con_neta"),
          F.count("cuantia_recuperada_num").alias("eventos_con_recuperacion")).show(truncate=False)
print("Netas negativas:", df.filter(F.col("cuantia_neta_num") < 0).count(), "| Brutas negativas:", df.filter(F.col("cuantia_bruta_num") < 0).count())

# Coherencia neta = bruta - recuperada (solo verificable donde están las tres cuantías)
chk = df.filter(F.col("cuantia_bruta_num").isNotNull() & F.col("cuantia_neta_num").isNotNull() & F.col("cuantia_recuperada_num").isNotNull())
print("Eventos con bruta+neta+recuperada:", chk.count())
display(chk.select("cuantia_bruta", "cuantia_total_recuperada", "cuantia_neta"))

# Severidad empírica (percentiles de la cuantía bruta, COP)
display(df.filter(F.col("cuantia_bruta_num").isNotNull())
      .select(F.percentile_approx("cuantia_bruta_num", [0.5, 0.9, 0.95, 0.99, 1.0]).alias("p50_p90_p95_p99_max")))

+-----------------+---------+---------+----------------+----------------+------------------------+
|eventos_con_bruta|bruta_min|bruta_max|bruta_suma      |eventos_con_neta|eventos_con_recuperacion|
+-----------------+---------+---------+----------------+----------------+------------------------+
|26701            |1.0      |7.0E9    |2.41789756996E11|26588           |235                     |
+-----------------+---------+---------+----------------+----------------+------------------------+

Netas negativas: 28 | Brutas negativas: 0
Eventos con bruta+neta+recuperada: 80


cuantia_bruta,cuantia_total_recuperada,cuantia_neta
301.200,300.000,1.200
4.650.174,4.646.424,3.750
2.212.114,2.182.600,29.514
2.800.940,2.769.080,31.860
2.754.573,2.732.800,21.773
1.096.922,548.461,548.461
648.000,324.000,324.000
6.771.806,5.247.900,1.523.906
200.800,200.000,800
5.563.769,5.500.000,63.769


p50_p90_p95_p99_max
"List(367000.0, 1.4197184E7, 3.0E7, 1.50124492E8, 7.0E9)"


In [0]:
# Variables categóricas clave (se observan variantes de mayúsculas/minúsculas y de tildes)
for colname in ["tipo", "codigoperdida", "moneda", "clase_de_riesgo", "categoria_n1", "fuente39", "fuente56", "linea_operativa_n1", "canal"]:
    print(f"-- {colname} --")
    d.groupBy(colname).count().orderBy(F.desc("count")).show(20, truncate=False)

-- tipo --
+----+-----+
|tipo|count|
+----+-----+
|P   |49820|
|R   |2067 |
|p   |2    |
+----+-----+

-- codigoperdida --
+-------------+-----+
|codigoperdida|count|
+-------------+-----+
|B            |49822|
|A            |2067 |
+-------------+-----+

-- moneda --
+------+-----+
|moneda|count|
+------+-----+
|PESOS |51870|
|Pesos |19   |
+------+-----+

-- clase_de_riesgo --
+-------------------+-----+
|clase_de_riesgo    |count|
+-------------------+-----+
|RIESGO OPERACIONAL |47913|
|RIESGO DE CONDUCTAS|3743 |
|RIESGO LAFT        |225  |
|RIESGO DE CRÉDITO  |6    |
|RIESGO DE LIQUIDEZ |2    |
+-------------------+-----+

-- categoria_n1 --
+---------------------------------------------+-----+
|categoria_n1                                 |count|
+---------------------------------------------+-----+
|EJECUCIÓN Y ADMINISTRACIÓN DE PROCESOS       |23218|
|FALLAS TECNOLÓGICAS                          |20134|
|FRAUDE EXTERNO                               |5599 |
|CLIENTES, PRODUCTOS Y

In [0]:
# REGLA DE NEGOCIO PARA EL VaR (20-Nov-2026): únicamente eventos con CODIGOPERDIDA = 'A'.
# Nota: en Bronze los valores crudos traen espacios ('R ' en 108 filas, y 2 filas de A no
# coinciden con 'A' exacto); el filtro se aplica sobre el valor normalizado (trim).
va = df.filter(F.col("codigoperdida") == "A")

print("Eventos con CODIGOPERDIDA='A':", va.count(), "de", df.count(), "totales")
va.agg(
    F.count("*").alias("eventos_A"),
    F.count("cuantia_bruta_num").alias("con_cuantia"),
    F.min("cuantia_bruta_num").alias("bruta_min"),
    F.max("cuantia_bruta_num").alias("bruta_max"),
    F.sum("cuantia_bruta_num").alias("bruta_suma")).show(truncate=False)

print("Percentiles de severidad bruta (solo A):")
display(va.filter(F.col("cuantia_bruta_num").isNotNull()).select(
    F.percentile_approx("cuantia_bruta_num", [0.5, 0.9, 0.99, 1.0]).alias("p50_p90_p99_max")))

print("Eventos A por año:")
display(va.groupBy("anio").count().orderBy("anio"))

print("Eventos A por categoría Basel:")
display(va.groupBy("categoria_n1").count().orderBy(F.desc("count")))

print("Recuperación registrada en A:", va.filter(F.col("cuantia_recuperada_num").isNotNull()).count(),
      "| Netas negativas en A:", va.filter(F.col("cuantia_neta_num") < 0).count())

Eventos con CODIGOPERDIDA='A': 2067 de 51889 totales
+---------+-----------+---------+------------+------------+
|eventos_A|con_cuantia|bruta_min|bruta_max   |bruta_suma  |
+---------+-----------+---------+------------+------------+
|2067     |2046       |4.0      |5.01334276E8|5.33517436E9|
+---------+-----------+---------+------------+------------+

Percentiles de severidad bruta (solo A):


p50_p90_p99_max
"List(113925.0, 3071200.0, 5.073E7, 5.01334276E8)"


Eventos A por año:


anio,count
2014,1
2015,147
2016,106
2017,217
2018,211
2019,141
2020,266
2021,159
2022,173
2023,151


Eventos A por categoría Basel:


categoria_n1,count
EJECUCIÓN Y ADMINISTRACIÓN DE PROCESOS,1140
FRAUDE EXTERNO,611
FALLAS TECNOLÓGICAS,226
RELACIONES LABORALES Y SEGURIDAD LABORAL,44
FRAUDE INTERNO,24
DAÑOS A ACTIVOS FÍSICOS,17
"CLIENTES, PRODUCTOS Y PRÁCTICAS EMPRESARIALES",5


Recuperación registrada en A: 235 | Netas negativas en A: 28


## Hallazgos verificados de la inspección

- **Volumen:** 51.889 filas × 58 columnas. 1 fila = 1 evento de riesgo operativo.
- **Filas con valores idénticos:** 1.031 filas comparten todos sus valores con otra fila (medición sobre datos normalizados en memoria; 1.029 sobre valores crudos). Por la estructura de la BD NO se consideran duplicadas (aclaración del negocio, 20-Nov-2026): cada fila es un registro independiente. IDINSUMO presente solo en 28.863 filas (28.351 distintos) -> no sirve como clave única.
- **Fechas:** eventos desde 2014-12-05 hasta 2026-09-30; todas las fechas de inicio se pudieron parsear. Contabilización y recuperación nulas en ~96-99,5%.
- **Montos:** 26.701 eventos con cuantía bruta (48,5% sin cuantificar); bruta mínima 1 COP, máxima 7.000.000.000 COP, suma ~241.790 millones COP. 28 netas negativas. Recuperación registrada solo en 235 eventos.
- **Categóricas:** TIPO alineado con CODIGOPERDIDA (P/p↔B = 49.822; R↔A = 2.067); MONEDA solo PESOS (con variante de mayúsculas); CANAL tiene 13 formas de escritura -> 8 canales reales; CLASE DE RIESGO en 5 clases (92,3% riesgo operacional).
- **Regla para el VaR (negocio, 20-Nov-2026):** el VaR se calcula únicamente con eventos CODIGOPERDIDA='A' (2.067 de 51.889, equivalen a TIPO='R'): 2.046 con cuantía bruta (21 sin cuantía, ~1%), severidad entre 4 y 501.334.276 COP (suma ~5.335 millones COP), p50=113.925 y p99≈50,7 millones COP. Todas las recuperaciones registradas (235) y todas las netas negativas (28) están en A. Los eventos B (49.822) quedan para análisis de frecuencia y descriptivos, no para el VaR.
- **Limitaciones para VaR (sobre el subconjunto A):** sin datos de exposición, umbral de reporte desconocido (sesgo de truncamiento en severidad), 2.046 observaciones de severidad en 13 años (año 2026 incompleto y 2014 con 1 evento A), y 524 códigos de causa sin nombre descriptivo.